In [1]:
import pandas as pd
from weather_pipeline.datasource import build_data_source

era5 = build_data_source("era5")
times = pd.date_range("2020-08-01", "2020-08-31T18:00", freq="6h")
coarse = era5(list(times), ["t2m"]).sel(variable="t2m", drop=True)

In [2]:
from huggingface_hub import hf_hub_download
from downscale.downscaler import Downscaler
from downscale.station_interp import GridInterpolator

model = Downscaler.load(hf_hub_download("EmmaScharfmann/france-temperature-downscaler", "model.pt"))
interp = GridInterpolator.build(coarse.lat, coarse.lon, model.lat.ravel(), model.lon.ravel())
on_fine = interp.apply(coarse).values.reshape(len(times), *model.lat.shape)
fine = model.predict(on_fine, times.to_numpy(), members=4)

In [4]:
import numpy as np
from downscale.publish import load_published_target

USE_HUB = True  # False to read a copy you downloaded from the Copernicus CDS

if USE_HUB:
    cerra, static = load_published_target("EmmaScharfmann/cerra-france-t2m-2016-2020")
else:
    cerra, static = load_published_target("data/cerra")

truth = cerra.t2m.sel(time=times).values
for name, field in [("coarse", on_fine), ("downscaled", fine.mean(1))]:
    print(name, np.sqrt(((field - truth) ** 2).mean()).round(3), "K")

coarse 1.47 K
downscaled 1.242 K


In [6]:
import xarray as xr
from downscale.diagnostics import comparison_figure, land_mask

static = xr.open_zarr("hf://datasets/EmmaScharfmann/cerra-france-t2m-2016-2020/static.zarr")
i = 25
print(comparison_figure(on_fine[i], fine[i].mean(0), truth[i], model.lat, model.lon,
                        "comparison.png", mask=land_mask(static),
                        time_label=str(times[i])))

{'coarse': 1.8078416585922241, 'downscaled': 1.2869890928268433}


In [7]:
from downscale.diagnostics import comparison_figure, land_mask

i = 25
print(comparison_figure(on_fine[i], fine[i].mean(0), truth[i], model.lat, model.lon,
                        "comparison.png", mask=land_mask(static), time_label=str(times[i])))

{'coarse': 1.8078416585922241, 'downscaled': 1.2869890928268433}


In [8]:
import numpy as np, xarray as xr
from downscale.stations import MeteoFranceHourly, load_published_stations, load_stations
from downscale.station_interp import curvilinear_for

USE_CACHE = True  # False to download the observations from Météo-France instead

if USE_CACHE:
    stations = load_published_stations("EmmaScharfmann/france-stations-2020-08")
else:
    stations, _ = load_stations(MeteoFranceHourly(), "france", times[0], times[-1])

read = curvilinear_for(model.lat, model.lon, stations).apply
obs = stations.t2m.sel(time=times).values
for name, field in [("coarse", on_fine), ("downscaled", fine.mean(1))]:
    at_stations = read(xr.DataArray(field, dims=("time", "y", "x"))).values
    print(name, np.sqrt(np.nanmean((at_stations - obs) ** 2)).round(3), "K")

ImportError: cannot import name 'load_published_stations' from 'downscale.stations' (/Users/emma/Code/HuggingFace/downscaling/weather-forecast-model-downscaler/downscale/stations.py)

In [ ]:
from downscale.diagnostics import floor_figure, score_against_stations

at_stations = {name: read(xr.DataArray(f, dims=("time", "y", "x"))).values
               for name, f in [("coarse", on_fine), ("downscaled", fine.mean(1)), ("target", truth)]}
scores = score_against_stations(at_stations, obs, stations.elevation.values,
                                times.hour.values, [0, 200, 500, 1000, 1500, 2500, 5000])
floor_figure(scores, "floor.png")
print(scores[scores.grouping == "all"][["field", "rmse_K"]].to_string(index=False))